[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-customer.ipynb)

# Full Project: Marketing Customer Value & Engagement Analysis

*AIBits Academy · Machine Learning End To End · Full Project*

A complete, business-driven EDA and segmentation walkthrough — turning a raw 24-column marketing dataset into an actionable targeting insight, using nothing more exotic than groupby and a median split.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> An auto-insurance company runs renewal marketing campaigns (agent calls, branch visits, call-centre outreach, web offers) but doesn't know which customers are actually worth targeting. Marketing spend is limited; blasting offers to the entire base is wasteful. The goal: use existing customer data to find out *which segments respond*, so campaigns can be focused where they convert.

> **Dataset**
>
> IBM Watson Analytics — Marketing Customer Value Analysis. **9,134 rows × 24 columns.** One row per auto-insurance policyholder: demographics (State, Gender, Employment Status, Income), policy details (Coverage, Policy Type, Monthly Premium, Vehicle Class/Size), engagement history (Months Since Policy Inception, Number of Open Complaints, Sales Channel), and the target column `Response` (Yes/No — did the customer respond to the current renewal offer?).

## Step 1 — Load and Inspect

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv')
print(df.shape)
df.head()

## Step 2 — Overall Engagement Rate

Before segmenting anything, establish the baseline: what fraction of customers respond to a renewal offer at all?

In [ ]:
# Count responders vs non-responders
counts = df.groupby('Response').count()['Customer']
print(counts)
print(counts / df.shape[0])

Only **14.3%** of customers respond to a renewal offer. Any segment that beats this baseline meaningfully is worth targeting; anything below it is worth deprioritising.

## Step 3 — Engagement by Renewal Offer Type & Sales Channel

In [ ]:
# Engagement rate per offer type
by_offer = (
    df.loc[df['Response'] == 'Yes'].groupby('Renew Offer Type').count()['Customer']
    / df.groupby('Renew Offer Type').count()['Customer']
)
print((by_offer * 100).round(2))

# Engagement rate per sales channel
by_channel = (
    df.loc[df['Response'] == 'Yes'].groupby('Sales Channel').count()['Customer']
    / df.groupby('Sales Channel').count()['Customer']
)
print((by_channel * 100).round(2))

**Offer2** converts best (23.4% vs the 14.3% baseline) and **Agent** outreach beats every self-service channel by a wide margin (19.2% vs ~11%). Two channels, already worth reallocating budget toward — but this is still an aggregate view. The real question is *which customers*, not just which channel.

## Step 4 — Feature Engineering: Turning Continuous Columns into Actionable Segments

Two columns carry a lot of signal but are hard to act on as raw numbers: `Customer Lifetime Value` (highly skewed, ranges from $1,898 to $83,325) and `Months Since Policy Inception` (0–99 months). A median-split turns each into a business-readable High/Low segment — exactly the kind of feature engineering that makes a model (or a marketing dashboard) usable by non-technical stakeholders.

In [ ]:
print(df['Customer Lifetime Value'].describe())

# Median-split feature engineering
clv_median = df['Customer Lifetime Value'].median()
df['CLV Segment'] = df['Customer Lifetime Value'].apply(
    lambda x: 'High' if x > clv_median else 'Low'
)

age_median = df['Months Since Policy Inception'].median()
df['Policy Age Segment'] = df['Months Since Policy Inception'].apply(
    lambda x: 'High' if x > age_median else 'Low'
)

## Step 5 — The Insight: Cross-Tabulating the Two Segments

In [ ]:
engagement_by_segment = df.loc[
    df['Response'] == 'Yes'
].groupby(['CLV Segment', 'Policy Age Segment']).count()['Customer'] / df.groupby(
    ['CLV Segment', 'Policy Age Segment']
).count()['Customer']

print((engagement_by_segment * 100).round(2))

> **The Business Insight**
>
> The highest-responding segment is **not** high-value, long-tenure customers — the group a naive "target your best customers" strategy would pick. It's **long-tenured, low-CLV** policyholders (16.25% response, more than double the 14.3% baseline). High-CLV customers respond *worse* regardless of tenure (13.2–13.9%), possibly because they already feel adequately served or are more price-sensitive to renewal terms. Without engineering the two segment columns, this inversion was completely invisible in the raw 24-column table — it only appears once continuous variables are converted into business-interpretable buckets and cross-tabulated.

## Visualizing the Segments

Left: engagement rate against the 14.3% baseline (dashed line), toggle between offer type and sales channel. Right: the 2×2 CLV×Tenure grid from Step 5 — darker cells engage more, and the winning cell is called out directly.

## Key Business Takeaways

- Aggregate engagement rate (14.3%) hides enormous segment-level variance (13.2% to 23.4% depending on offer type, channel, and CLV/tenure segment).
- Agent-led outreach dramatically outperforms self-service channels for this population — a resourcing signal, not just an analytics footnote.
- Feature engineering (median-split segmentation) turned two noisy continuous columns into a business-actionable 2×2 targeting grid.
- The most counter-intuitive finding — that low-CLV, long-tenure customers convert best — is exactly the kind of insight that pure eyeballing of raw columns would miss, and that a stakeholder can act on immediately without needing a trained model in production.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The baseline response rate

Store in `resp_rate` the share of customers whose `Response` is `"Yes"`, as a fraction between 0 and 1.

In [ ]:
resp_rate = None   # TODO


In [ ]:
try:
    check("about 14.3%", abs(resp_rate - 1308 / 9134) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
resp_rate = (df["Response"] == "Yes").mean()

```

</details>

### Exercise 2 · Medium · Response by coverage level

Compute `by_cov`: the response rate (fraction) for each `Coverage` level as a Series sorted from highest to lowest.

In [ ]:
by_cov = None   # TODO


In [ ]:
try:
    ref = df.groupby("Coverage")["Response"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
    check("same order", list(by_cov.index) == list(ref.index))
    check("same values", (abs(by_cov.values - ref.values) < 1e-12).all())
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
by_cov = df.groupby("Coverage")["Response"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)

```

</details>

### Exercise 3 · Stretch · Find the best segment cell

Split `Income` at its median into `"High"`/`"Low"` (`income_seg`, like the lesson's CLV split) and cross it with `Sales Channel`. Store the response-rate table in `seg_table` (rows = income segment, columns = channel) and the `(income_seg, channel)` pair with the highest rate in `best_cell`.

In [ ]:
seg_table = best_cell = None   # TODO


In [ ]:
try:
    check("2 x 4 table", seg_table.shape == (2, 4))
    check("best cell is the max", abs(seg_table.loc[best_cell[0], best_cell[1]] - seg_table.values.max()) < 1e-12)
    check("rates are fractions", ((seg_table >= 0) & (seg_table <= 1)).all().all())
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
df["income_seg"] = (df["Income"] > df["Income"].median()).map({True: "High", False: "Low"})
seg_table = df.assign(yes=(df["Response"] == "Yes")).pivot_table(index="income_seg", columns="Sales Channel", values="yes", aggfunc="mean")
best_cell = seg_table.stack().idxmax()

```

Segmentation like this turns a model-free table into a targeting rule the marketing team can act on.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Marketing Customer Value & Engagement Analysis**.*